# Road Inspector AI Training
Run these cells to generate the dataset and train the ML models.

In [ ]:
import pandas as pd
import numpy as np

# 1. Define Parameters & Ranges
num_samples = 50000
np.random.seed(42)

area_sqm = np.random.uniform(1.0, 50.0, num_samples)
depth_m = np.random.choice([0.03, 0.04, 0.05, 0.075, 0.1], num_samples)
ambient_temp_c = np.random.uniform(22.0, 40.0, num_samples)
transport_time_hr = np.random.uniform(0.5, 4.0, num_samples)
humidity_pct = np.random.uniform(40, 95, num_samples)

density_hma = 2.35
wastage_factor = 1.10
bitumen_ratio = 0.05

volume_m3 = area_sqm * depth_m
hma_tonnes = volume_m3 * density_hma * wastage_factor
bitumen_liters = (hma_tonnes * 1000) * bitumen_ratio

labor_hours = volume_m3 * 2.5
crew_size = np.ceil(labor_hours / 8).clip(min=1) 
machine_hours = volume_m3 * 1.5

labor_cost = crew_size * 8500
machine_cost = machine_hours * 15000
material_cost = hma_tonnes * 28000
total_cost = labor_cost + machine_cost + material_cost

target_arrival_temp = 145.0
dispatch_temp = target_arrival_temp + (transport_time_hr * (150 - ambient_temp_c) * 0.15)
compaction_window_min = (ambient_temp_c / target_arrival_temp) * 60 + np.random.uniform(-5, 5, num_samples)

df = pd.DataFrame({
    'area_sqm': area_sqm,
    'depth_m': depth_m,
    'ambient_temp_c': ambient_temp_c,
    'transport_time_hr': transport_time_hr,
    'humidity_pct': humidity_pct,
    'hma_tonnes': hma_tonnes,
    'bitumen_liters': bitumen_liters,
    'labor_hours': labor_hours,
    'machine_hours': machine_hours,
    'dispatch_temp_c': dispatch_temp,
    'compaction_window_min': compaction_window_min,
    'total_cost_lkr': total_cost
})

df.to_csv('road_repair_dataset.csv', index=False)
print(f"Dataset generated successfully with {len(df)} records!")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import joblib

data = pd.read_csv('road_repair_dataset.csv')
X = data[['area_sqm', 'depth_m', 'ambient_temp_c', 'transport_time_hr', 'humidity_pct']]
y = data[['hma_tonnes', 'bitumen_liters', 'labor_hours', 'machine_hours', 'total_cost_lkr']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

predictions = rf_model.predict(X_test)
r2 = r2_score(y_test, predictions)
print(f"Model Accuracy (R² Score): {r2:.4f}")

joblib.dump(rf_model, 'rf_resource_model.pkl')
print("Model saved as 'rf_resource_model.pkl'")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

X_therm = data[['ambient_temp_c', 'transport_time_hr', 'humidity_pct', 'depth_m']]
y_therm = data[['dispatch_temp_c', 'compaction_window_min']]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_therm, y_therm, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_t)
X_test_scaled = scaler.transform(X_test_t)

print("Training MLP Neural Network...")
mlp_model = MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=500, random_state=42)
mlp_model.fit(X_train_scaled, y_train_t)

mlp_preds = mlp_model.predict(X_test_scaled)
r2_therm = r2_score(y_test_t, mlp_preds)
print(f"Thermodynamic Model Accuracy (R² Score): {r2_therm:.4f}")

joblib.dump(scaler, 'therm_scaler.pkl')
joblib.dump(mlp_model, 'mlp_thermal_model.pkl')
print("Models saved as 'therm_scaler.pkl' and 'mlp_thermal_model.pkl'")

In [ ]:
from google.colab import files

files.download('road_repair_dataset.csv')
files.download('rf_resource_model.pkl')
files.download('therm_scaler.pkl')
files.download('mlp_thermal_model.pkl')